# TP-1: CleaningData - Applying OOP and Design Patterns

## Exercise 1: Build the CSVReader Class (OOP Foundation)

#### Instructions:
1.	Create a class CSVReader.
2.	Define an __init__ method that takes file_path.
3.	Add a read() method that loads the CSV file using pandas.
4.	Add a preview(n) method that prints the top n rows.


In [10]:
import pandas as pd
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    def read(self):
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        if self.data is None:
            self.read()
        print(self.data.head(n))
        return self.data.head(n)


# Test Exercise 1
reader = CSVReader("sample_data (2).csv")
df = reader.read()

print("Dataset:")
reader.preview(10)

Dataset:
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0      Chicago   92.0


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


## Exercise 2: Apply Pattern for Missing Values Handling

1.	Create an abstract class MissingValueStrategy with a method handle(df).
2.	Create subclasses:
o	DropMissing
o	FillMean
o	FillMode
3.	Create a DataCleaner class that applies the chosen strategy.


In [9]:
from abc import ABC, abstractmethod


class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df):
        pass


class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna().copy()


class FillMean(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()

        numeric_columns = result.select_dtypes(include="number").columns

        for column in numeric_columns:
            result[column] = result[column].fillna(
                result[column].mean()
            )

        return result


class FillMode(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()

        for column in result.columns:
            mode_values = result[column].mode(dropna=True)

            if not mode_values.empty:
                result[column] = result[column].fillna(
                    mode_values.iloc[0]
                )

        return result


class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)


# Test Exercise 2
print("Missing values before cleaning:")
print(df.isnull().sum())

cleaner = DataCleaner(FillMean())
df_cleaned = cleaner.clean(df)

print("\nMissing values after FillMean:")
print(df_cleaned.isnull().sum())

print("\nCleaned dataset:")
print(df_cleaned)

Missing values before cleaning:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64

Missing values after FillMean:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64

Cleaned dataset:
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0


## Exercise 3: Add Decorators for Logging and Timing

1.	Create two decorators:
- @log_action → logs when a method starts and finishes.
- @log_time → calculates and prints execution time.
2.	Apply them to methods in CSVReader or DataCleaner.


In [5]:
from functools import wraps
from time import perf_counter


def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Starting: {func.__name__}")

        try:
            return func(*args, **kwargs)
        finally:
            print(f"Finished: {func.__name__}")

    return wrapper


def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = perf_counter()

        try:
            return func(*args, **kwargs)
        finally:
            end_time = perf_counter()
            elapsed = end_time - start_time

            print(f"Execution time: {elapsed:.6f} seconds")

    return wrapper


# Apply decorators to DataCleaner.clean()
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    @log_action
    @log_time
    def clean(self, df):
        return self.strategy.handle(df)


# Test Exercise 3
cleaner = DataCleaner(FillMean())
df_cleaned = cleaner.clean(df)

print(df_cleaned)

Starting: clean
Execution time: 0.001986 seconds
Finished: clean
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0


## Exercise 4: Implement Factory Pattern for Data Transformations

1.	Create an abstract class DataTransform with a method apply(df).
2.	Implement subclasses:
- NormalizeColumns
- RemoveDuplicates
- StandardizeText
3.	Create a TransformFactory that returns transformation objects based on string input.


In [6]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df):
        pass


class NormalizeColumns(DataTransform):
    def apply(self, df):
        result = df.copy()

        numeric_columns = result.select_dtypes(
            include="number"
        ).columns

        for column in numeric_columns:
            # Do not normalize the ID column
            if column.lower() == "id":
                continue

            minimum = result[column].min()
            maximum = result[column].max()

            if pd.isna(minimum) or pd.isna(maximum):
                continue

            if maximum == minimum:
                result[column] = result[column].where(
                    result[column].isna(), 0.0
                )
            else:
                result[column] = (
                    result[column] - minimum
                ) / (maximum - minimum)

        return result


class RemoveDuplicates(DataTransform):
    def apply(self, df):
        return df.drop_duplicates().copy()


class StandardizeText(DataTransform):
    def apply(self, df):
        result = df.copy()

        text_columns = result.select_dtypes(
            include=["object", "string"]
        ).columns

        for column in text_columns:
            result[column] = (
                result[column]
                .astype("string")
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
                .str.title()
            )

        return result


class TransformFactory:
    @staticmethod
    def create_transform(transform_name):
        transformations = {
            "normalize": NormalizeColumns,
            "remove_duplicates": RemoveDuplicates,
            "standardize_text": StandardizeText
        }

        name = transform_name.strip().lower()

        if name not in transformations:
            raise ValueError(
                f"Unknown transformation: {transform_name}"
            )

        return transformations[name]()


# Test Exercise 4
transform = TransformFactory.create_transform("normalize")
df_transformed = transform.apply(df_cleaned)

print("After normalization:")
print(df_transformed)

transform = TransformFactory.create_transform(
    "remove_duplicates"
)
df_transformed = transform.apply(df_transformed)

transform = TransformFactory.create_transform(
    "standardize_text"
)
df_transformed = transform.apply(df_transformed)

print("\nAfter all transformations:")
print(df_transformed)

After normalization:
   id     name      age  height_cm  weight_kg         city     score
0   1    Alice  0.25000   0.333333   0.433333     New York  0.588235
1   2      Bob  0.40625   0.800000   0.500000  Los Angeles  0.882353
2   3  Charlie  1.00000   0.533333   0.566667      Chicago  0.647059
3   4    David  0.40625   0.466667   0.833333      Houston  0.000000
4   5      Eva  0.00000   0.000000   0.000000     New York  0.764706
5   6    Frank  0.37500   1.000000   1.000000  Los Angeles  0.647059
6   7    Grace  0.40625   0.133333   0.166667      Chicago  1.000000

After all transformations:
   id     name      age  height_cm  weight_kg         city     score
0   1    Alice  0.25000   0.333333   0.433333     New York  0.588235
1   2      Bob  0.40625   0.800000   0.500000  Los Angeles  0.882353
2   3  Charlie  1.00000   0.533333   0.566667      Chicago  0.647059
3   4    David  0.40625   0.466667   0.833333      Houston  0.000000
4   5      Eva  0.00000   0.000000   0.000000     New 

## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

1.	Create an abstract class DataPipeline with a run() method defining steps:
- load()
- clean()
- transform()
- save()
2.	Create CSVDataPipeline that implements each step using previous classes.


In [7]:
class DataPipeline(ABC):

    # Template Method
    def run(self):
        self.load()
        self.clean()
        self.transform()
        self.save()

        print("\nPipeline completed successfully!")

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self):
        pass

    @abstractmethod
    def transform(self):
        pass

    @abstractmethod
    def save(self):
        pass


class CSVDataPipeline(DataPipeline):

    def __init__(self, input_path, output_path):
        self.input_path = input_path
        self.output_path = output_path
        self.data = None

    def load(self):
        print("\nStep 1: Loading dataset")

        reader = CSVReader(self.input_path)
        self.data = reader.read()

        print(self.data.head())

    def clean(self):
        print("\nStep 2: Handling missing values")

        cleaner = DataCleaner(FillMean())
        self.data = cleaner.clean(self.data)

        print(self.data.isnull().sum())

    def transform(self):
        print("\nStep 3: Transforming dataset")

        transformations = [
            "normalize",
            "remove_duplicates",
            "standardize_text"
        ]

        for name in transformations:
            transform = TransformFactory.create_transform(name)
            self.data = transform.apply(self.data)

            print(f"Applied: {name}")

    def save(self):
        print("\nStep 4: Saving dataset")

        self.data.to_csv(self.output_path, index=False)

        print(f"Dataset saved to: {self.output_path}")
        print(f"Final shape: {self.data.shape}")


# Run Exercise 5
pipeline = CSVDataPipeline(
    input_path="sample_data (2).csv",
    output_path="cleaned_data.csv"
)

pipeline.run()

print("\nFinal dataset:")
print(pipeline.data.head())


Step 1: Loading dataset
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0

Step 2: Handling missing values
Starting: clean
Execution time: 0.002197 seconds
Finished: clean
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64

Step 3: Transforming dataset
Applied: normalize
Applied: remove_duplicates
Applied: standardize_text

Step 4: Saving dataset
Dataset saved to: cleaned_data.csv
Final shape: (7, 7)

Pipeline completed successfully!

Final dataset:
   id     name      age  height_cm  weight_kg         city     score
0   1    Alice  0.25000   0.333333   0.433333     New York  0.588235
1   2      B